In [ ]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

NOAA_API  = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
NOAA_META = "https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/{}.json"
YEAR = 2026

In [ ]:
STATION_IDS = ["8518750", "8516945"]  # The Battery, Kings Point


def station_meta(station_id):
    r = requests.get(NOAA_META.format(station_id))
    r.raise_for_status()
    s = r.json()["stations"][0]
    return {"name": s["name"], "lat": float(s["lat"]), "lng": float(s["lng"])}


def fetch_predictions(station_id, year):
    """Fetch high/low predictions one month at a time to avoid API timeouts."""
    months = []
    for month in range(1, 13):
        begin = pd.Timestamp(year, month, 1)
        end   = begin + pd.offsets.MonthEnd(0)
        r = requests.get(NOAA_API, params={
            "product":    "predictions",
            "begin_date": begin.strftime("%Y%m%d"),
            "end_date":   end.strftime("%Y%m%d"),
            "datum":      "MLLW",
            "station":    station_id,
            "time_zone":  "lst_ldt",
            "interval":   "hilo",
            "units":      "english",
            "format":     "json",
        })
        r.raise_for_status()
        months.append(pd.DataFrame(r.json()["predictions"]))
    return pd.concat(months, ignore_index=True)


frames = []
for sid in STATION_IDS:
    meta = station_meta(sid)
    df   = fetch_predictions(sid, YEAR)

    df["datetime"]     = pd.to_datetime(df["t"], format="%Y-%m-%d %H:%M")
    df["pred_in_ft"]   = df["v"].astype(float)
    df["pred_in_cm"]   = (df["pred_in_ft"] * 30.48).round().astype(int)
    df["date"]         = df["datetime"].dt.strftime("%Y/%m/%d")
    df["day"]          = df["datetime"].dt.strftime("%a")
    df["time"]         = df["datetime"].dt.strftime("%I:%M %p")
    df["highlow"]      = df["type"]
    df["station_id"]   = sid
    df["station_name"] = meta["name"]
    df["geometry"]     = Point(meta["lng"], meta["lat"])

    frames.append(df[["datetime", "date", "day", "time",
                       "pred_in_ft", "pred_in_cm", "highlow",
                       "station_id", "station_name", "geometry"]])

tides_gdf = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326",
)

In [ ]:
for sid, group in tides_gdf.groupby("station_id", sort=False):
    print(f"\n── {group['station_name'].iloc[0]}  ({sid})  "
          f"{group['geometry'].iloc[0].y:.4f}°N, {group['geometry'].iloc[0].x:.4f}°W ──")
    display(group[["date", "day", "time", "pred_in_ft", "pred_in_cm", "highlow"]]
              .head(8)
              .reset_index(drop=True))

print(f"\nTotal records: {len(tides_gdf)}  |  CRS: {tides_gdf.crs}")

tides_gdf.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\tide_predictions_2026.geojson",
    driver="GeoJSON",
)
print("Saved → tide_predictions_2026.geojson")

## USGS Tidal Observations (parameter 62620 — water surface elevation above NAVD 1988)

In [ ]:
import dataretrieval.nwis as nwis

USGS_SITES = {
    "01311875": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01311850": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01311145": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01376562": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet
}

START_DT = "2026-01-01"
END_DT = pd.Timestamp.today().strftime("%Y-%m-%d")


def fetch_usgs_meta(site_no):
    df = nwis.get_record(sites=site_no, service="site")
    row = df.iloc[0]
    return {
        "name": row["station_nm"],
        "lat":  float(row["dec_lat_va"]),
        "lon":  float(row["dec_long_va"]),
    }


def fetch_usgs_wl(site_no, param, start_dt, end_dt):
    df, _ = nwis.get_iv(sites=site_no, parameterCd=param, startDT=start_dt, endDT=end_dt)
    value_col = next(
        c for c in df.columns
        if (c == param or c.startswith(param + "_")) and not c.endswith("_cd")
    )
    df = df[[value_col]].rename(columns={value_col: "value"})
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df.index.name = "datetime"
    return df.reset_index()


frames = []
for site_no, param_cd in USGS_SITES.items():
    meta = fetch_usgs_meta(site_no)
    df   = fetch_usgs_wl(site_no, param_cd, START_DT, END_DT)
    df["site_no"]      = site_no
    df["param_cd"]     = param_cd
    df["station_name"] = meta["name"]
    df["geometry"]     = Point(meta["lon"], meta["lat"])
    frames.append(df)

usgs_gdf = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326",
)

In [ ]:
for site, group in usgs_gdf.groupby("site_no", sort=False):
    print()
    print(f"── {group['station_name'].iloc[0]}  ({site})  param {group['param_cd'].iloc[0]}  "
          f"{group['geometry'].iloc[0].y:.4f}°N, {group['geometry'].iloc[0].x:.4f}°W ──")
    display(group[["datetime", "value"]].head(8).reset_index(drop=True))

print()
print(f"Total records: {len(usgs_gdf)}  |  CRS: {usgs_gdf.crs}")

usgs_gdf.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\usgs_water_levels.geojson",
    driver="GeoJSON",
)
print("Saved → usgs_water_levels.geojson")

## Standardized Unified Schema

Both sources are aligned to a common schema:

| column | type | notes |
|---|---|---|
| `datetime` | datetime (UTC, tz-aware) | NOAA localized from US/Eastern; USGS converted to UTC |
| `water_level_ft` | float | **NAVD88 datum** for both sources |
| `tidal_phase` | string | `H` / `L` (NOAA); `rising` / `falling` (USGS) |
| `surge_ft` | float | `0.0` for NOAA predicted; `NaN` for USGS (requires paired predicted) |
| `source` | string | `"noaa_predicted"` or `"usgs_observed"` |
| `station_id` | string | NOAA station ID or USGS site number |
| `station_name` | string | — |
| `geometry` | Point (EPSG:4326) | — |

**Datum conversion (NOAA only):** NOAA predictions are in feet above MLLW. NOAA's own datums API gives the height of each datum above the station datum (STND); the difference `MLLW_stnd − NAVD_stnd` equals how far MLLW is above NAVD88. Subtracting that offset converts every NOAA reading to feet above NAVD88.

In [ ]:
def fetch_noaa_datums(station_id):
    r = requests.get(
        f"https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/{station_id}/datums.json",
        params={"units": "english"},
    )
    r.raise_for_status()
    return {d["name"]: float(d["value"]) for d in r.json()["datums"]}


# How far MLLW sits above NAVD88 at each station (feet).
# Negative at NYC stations — MLLW is below NAVD88.
# Adding this to a MLLW-referenced value gives a NAVD88-referenced value.
MLLW_ABOVE_NAVD = {}
for sid in STATION_IDS:
    d = fetch_noaa_datums(sid)
    MLLW_ABOVE_NAVD[sid] = d["MLLW"] - d["NAVD88"]
    print(f"{sid}: MLLW is {MLLW_ABOVE_NAVD[sid]:+.3f} ft above NAVD88")

### Datum conversion: MLLW → NAVD88

**What the NOAA datums API returns**

`fetch_noaa_datums()` returns every datum for a station as a height **above STND** — the physical benchmark bolt at the gauge. STND is just a local reference pin; it has no global meaning. Both `d["MLLW"]` and `d["NAVD88"]` are measured up from the same STND, so their difference is a meaningful physical offset between the two datums.

**Physical ordering at NYC stations**

NAVD88 approximates mean sea level. MLLW (Mean Lower Low Water) is the average of the lowest daily tides — well below mean sea level. So at every NYC gauge:

```
↑  MHW    (~+2 ft relative to NAVD88)
↑  NAVD88  ≈ mean sea level          d["NAVD88"] > d["MLLW"]
↑  MLLW   (~-2.5 ft relative to NAVD88)
```

Therefore `MLLW_ABOVE_NAVD = d["MLLW"] − d["NAVD88"]` is a **negative number** — it quantifies how far below NAVD88 the MLLW zero mark sits.

**Converting a MLLW-referenced prediction to NAVD88**

NOAA tide predictions (`pred_in_ft`) are heights above the MLLW zero mark. To convert, find the water surface's absolute elevation above STND, then re-reference it to NAVD88:

```
absolute elevation above STND  =  d["MLLW"] + pred_in_ft
elevation above NAVD88         = (d["MLLW"] + pred_in_ft) − d["NAVD88"]
                               =  pred_in_ft + (d["MLLW"] − d["NAVD88"])
                               =  pred_in_ft + MLLW_ABOVE_NAVD
```

Because `MLLW_ABOVE_NAVD` is negative, this **subtracts** the gap between NAVD88 and MLLW — which is exactly what you want. A low-tide reading of 0.3 ft above MLLW becomes `0.3 + (−2.5) = −2.2 ft` relative to NAVD88, correctly placing low water below mean sea level.

**Why `− MLLW_ABOVE_NAVD` is wrong**

Subtracting a negative number adds its magnitude: `pred − (−2.5) = pred + 2.5`. That pushes every reading *further above* MLLW rather than re-referencing it downward to NAVD88 — a physically impossible result for typical tide levels.

In [ ]:
import numpy as np

# ── NOAA: MLLW → NAVD88, datetimes localized US/Eastern → UTC ──
noaa_rows = []
for sid, group in tides_gdf.groupby("station_id", sort=False):
    g = group.copy()
    g["water_level_ft"] = g["pred_in_ft"] + MLLW_ABOVE_NAVD[sid]
    g["datetime"] = (
        g["datetime"]
        .dt.tz_localize("US/Eastern", ambiguous="NaT", nonexistent="shift_forward")
        .dt.tz_convert("UTC")
    )
    g = g.dropna(subset=["datetime"])
    g["tidal_phase"] = g["highlow"].map({"H": "H", "L": "L"})
    g["surge_ft"]    = 0.0
    g["source"]      = "noaa_predicted"
    g["station_id"]  = sid
    noaa_rows.append(g[["datetime", "water_level_ft", "tidal_phase",
                         "surge_ft", "source", "station_id", "station_name", "geometry"]])

noaa_std = gpd.GeoDataFrame(
    pd.concat(noaa_rows, ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)

# ── USGS: all sites use 62620 (NAVD88); normalize datetimes to UTC ──
usgs_rows = []
for site, group in usgs_gdf.groupby("site_no", sort=False):
    g = group.copy().sort_values("datetime").reset_index(drop=True)
    g["water_level_ft"] = g["value"]
    dt = pd.to_datetime(g["datetime"])
    g["datetime"] = dt.dt.tz_convert("UTC") if dt.dt.tz is not None else dt.dt.tz_localize("UTC")
    dv = g["water_level_ft"].diff()
    phase = pd.array([pd.NA] * len(g), dtype="string")
    phase[dv > 0] = "rising"
    phase[dv < 0] = "falling"
    g["tidal_phase"] = phase
    # Relabel peak transitions as H/L so downstream code can filter tidal_phase == 'H'
    next_phase = g["tidal_phase"].shift(-1)
    g.loc[(g["tidal_phase"] == "rising")  & (next_phase == "falling"), "tidal_phase"] = "H"
    g.loc[(g["tidal_phase"] == "falling") & (next_phase == "rising"),  "tidal_phase"] = "L"
    g["surge_ft"]    = np.nan
    g["source"]      = "usgs_observed"
    g["station_id"]  = site
    usgs_rows.append(g[["datetime", "water_level_ft", "tidal_phase",
                         "surge_ft", "source", "station_id", "station_name", "geometry"]])

usgs_std = gpd.GeoDataFrame(
    pd.concat(usgs_rows, ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)

# ── Combine ──
tidal_unified = gpd.GeoDataFrame(
    pd.concat([noaa_std, usgs_std], ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)

tidal_unified.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\tidal_unified_2026.geojson",
    driver="GeoJSON",
)

print(f"Schema: {list(tidal_unified.columns)}")
print(f"Sources: {tidal_unified['source'].value_counts().to_dict()}")
print(f"Total rows: {len(tidal_unified)}")
print(f"Datetime range: {tidal_unified['datetime'].min()} → {tidal_unified['datetime'].max()}")
display(tidal_unified.groupby("source").head(3).reset_index(drop=True))